In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Combined Deepfake Detector: Attention + Feature Selection\n",
    "\n",
    "This notebook implements the hybrid deepfake detection model by first writing all the necessary Python modules to the local environment (`model.py`, `utils.py`, `train.py`, `main.py`, `predict.py`) and then executing them.\n",
    "\n",
    "This workflow combines:\n",
    "1.  **Attention Mechanism (from 2022 paper):** Applied to the feature maps of all three backbones.\n",
    "2.  **Feature Stacking & Selection (from 2025 paper):** The attention-weighted features from all three models are stacked, and then a feature selection process (ReliefF, MI, mRMR) is used to find the best features for a final KNN classifier."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 0: Install Dependencies\n",
    "\n",
    "We need `joblib` to save the trained KNN model and scaler."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install joblib"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 1: Writing Project Files\n",
    "\n",
    "These cells use `%%writefile` to create the Python scripts in our environment."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### File 1: `model.py`\n",
    "\n",
    "This file defines all the Keras models. It includes the custom attention layers from the 2022 paper and the functions to create the three attention-based feature extractors (DenseNet121, EfficientNetB0, Xception)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "%%writefile model.py\n",
    "import tensorflow as tf\n",
    "from tensorflow.keras.layers import (\n",
    "    Dense,\n",
    "    Conv2D,\n",
    "    BatchNormalization,\n",
    "    Dropout,\n",
    "    Reshape,\n",
    "    Add,\n",
    "    Flatten,\n",
    ")\n",
    "from tensorflow.keras.models import Model\n",
    "import tensorflow.keras.applications.densenet as densenet\n",
    "import tensorflow.keras.applications.efficientnet as efficientnet\n",
    "import tensorflow.keras.applications.xception as xception\n",
    "\n",
    "# --- Custom Attention Layers (from deepfake-detection2/model.py) ---\n",
    "\n",
    "class ModifiedBranch(tf.keras.layers.Layer):\n",
    "    def __init__(self, a_vec_size, **kwargs):\n",
    "        super(ModifiedBranch, self).__init__(**kwargs)\n",
    "        self.a_vec_size = a_vec_size\n",
    "        self.dense_layer = Dense(self.a_vec_size, activation=\"tanh\", name=\"att_mod_dense\")\n",
    "\n",
    "    def call(self, input):\n",
    "        af = tf.keras.backend.mean(input, axis=2)\n",
    "        hs = self.dense_layer(af)\n",
    "        return hs\n",
    "\n",
    "    def get_config(self):\n",
    "        config = super().get_config()\n",
    "        config.update({\"a_vec_size\": self.a_vec_size})\n",
    "        return config\n",
    "\n",
    "\n",
    "class MainBranch(tf.keras.layers.Layer):\n",
    "    def __init__(self, a_vec_size, dim, **kwargs):\n",
    "        super(MainBranch, self).__init__(**kwargs)\n",
    "        self.a_vec_size = a_vec_size\n",
    "        self.dim = dim\n",
    "        self.reshape1 = Reshape((-1, self.a_vec_size), name=\"att_main_reshape1\")\n",
    "        self.relu = tf.keras.activations.relu\n",
    "        self.dropout = Dropout(0.5, name=\"att_main_dropout\")\n",
    "        self.reshape2 = Reshape((self.dim**2, self.a_vec_size), name=\"att_main_reshape2\")\n",
    "\n",
    "    def call(self, input):\n",
    "        e = tf.transpose(input, perm=[0, 2, 1])\n",
    "        e = self.reshape1(e)\n",
    "        e = self.relu(e)\n",
    "        e = self.dropout(e)\n",
    "        e = self.reshape2(e)\n",
    "        e = tf.transpose(e, perm=[0, 2, 1])\n",
    "        return e\n",
    "\n",
    "    def get_config(self):\n",
    "        config = super().get_config()\n",
    "        config.update({\"a_vec_size\": self.a_vec_size, \"dim\": self.dim})\n",
    "        return config\n",
    "\n",
    "\n",
    "class Attention(tf.keras.layers.Layer):\n",
    "    def __init__(self, dim, a_vec_size, **kwargs):\n",
    "        super(Attention, self).__init__(**kwargs)\n",
    "        self.dim = dim\n",
    "        self.a_vec_size = a_vec_size\n",
    "        self.dense1 = Dense(self.dim**2, name=\"att_att_dense1\")\n",
    "        self.reshape1 = Reshape((1, self.dim**2), name=\"att_att_reshape1\")\n",
    "        self.add = Add(name=\"att_att_add\")\n",
    "        self.dropout = Dropout(0.5, name=\"att_att_dropout\")\n",
    "        self.relu = tf.keras.activations.relu\n",
    "        self.reshape2 = Reshape((-1, self.a_vec_size), name=\"att_att_reshape2\")\n",
    "        self.dense2 = Dense(1, use_bias=False, name=\"att_att_dense2\")\n",
    "        self.reshape3 = Reshape((-1, self.dim**2), name=\"att_att_reshape3\")\n",
    "\n",
    "    def call(self, input):\n",
    "        eh = self.dense1(input[0])\n",
    "        eh = self.reshape1(eh)\n",
    "        eh = self.add([input[1], eh])\n",
    "        eh = self.relu(eh)\n",
    "        eh = self.dropout(eh)\n",
    "        eh = tf.transpose(eh, perm=[0, 2, 1])\n",
    "        eh = self.reshape2(eh)\n",
    "        eh = self.dense2(eh)\n",
    "        eh = self.reshape3(eh)\n",
    "        eh = self.relu(eh)\n",
    "        return eh\n",
    "\n",
    "    def get_config(self):\n",
    "        config = super().get_config()\n",
    "        config.update({\"dim\": self.dim, \"a_vec_size\": self.a_vec_size})\n",
    "        return config\n",
    "\n",
    "# --- Backbone Configuration and Extractor Creation ---\n",
    "\n",
    "def get_backbone_config(backbone_name, input_shape=(299, 299, 3)):\n",
    "    \"\"\"\n",
    "    Returns the correct pre-trained backbone, preprocessing function,\n",
    "    feature map layer, and dimensions.\n",
    "    \"\"\"\n",
    "    if backbone_name == \"DenseNet121\":\n",
    "        base_model = densenet.DenseNet121(\n",
    "            include_top=False, weights=\"imagenet\", input_shape=input_shape\n",
    "        )\n",
    "        feature_map_layer = base_model.layers[-2].output\n",
    "        preprocess_func = densenet.preprocess_input\n",
    "        dim = 9\n",
    "        a_vec_size = 1024\n",
    "    elif backbone_name == \"EfficientNetB0\":\n",
    "        base_model = efficientnet.EfficientNetB0(\n",
    "            include_top=False, weights=\"imagenet\", input_shape=input_shape\n",
    "        )\n",
    "        feature_map_layer = base_model.layers[-3].output\n",
    "        preprocess_func = efficientnet.preprocess_input\n",
    "        dim = 9\n",
    "        a_vec_size = 1280\n",
    "    elif backbone_name == \"Xception\":\n",
    "        base_model = xception.Xception(\n",
    "            include_top=False, weights=\"imagenet\", input_shape=input_shape\n",
    "        )\n",
    "        feature_map_layer = base_model.layers[-13].output\n",
    "        preprocess_func = xception.preprocess_input\n",
    "        dim = 19\n",
    "        a_vec_size = 1024\n",
    "    else:\n",
    "        raise ValueError(f\"Unknown backbone: {backbone_name}\")\n",
    "\n",
    "    base_model.trainable = False\n",
    "    return base_model, preprocess_func, feature_map_layer, dim, a_vec_size\n",
    "\n",
    "def create_attention_extractor(backbone_name, input_shape=(299, 299, 3)):\n",
    "    \"\"\"\n",
    "    Creates a model that applies the attention mechanism to the feature maps.\n",
    "    \"\"\"\n",
    "    base_model, _, feature_map_layer, dim, a_vec_size = get_backbone_config(\n",
    "        backbone_name, input_shape\n",
    "    )\n",
    "\n",
    "    x = Conv2D(\n",
    "        filters=a_vec_size,\n",
    "        kernel_size=(1, 1),\n",
    "        strides=(1, 1),\n",
    "        padding=\"valid\",\n",
    "        use_bias=True,\n",
    "        name=f\"{backbone_name}_att_conv\",\n",
    "    )(feature_map_layer)\n",
    "    x = BatchNormalization(axis=-1, name=f\"{backbone_name}_att_bn\")(x)\n",
    "    x = tf.keras.activations.relu(x)\n",
    "    x = Dropout(0.8, name=f\"{backbone_name}_att_drop\")(x)\n",
    "    x = Reshape((a_vec_size, dim**2), name=f\"{backbone_name}_att_reshape\")(x)\n",
    "\n",
    "    modified = ModifiedBranch(a_vec_size, name=f\"{backbone_name}_mod_branch\")(x)\n",
    "    main = MainBranch(a_vec_size, dim, name=f\"{backbone_name}_main_branch\")(x)\n",
    "    attention_features = Attention(dim, a_vec_size, name=f\"{backbone_name}_attention\")(\n",
    "        [modified, main]\n",
    "    )\n",
    "    output_features = Flatten(name=f\"{backbone_name}_flatten\")(attention_features)\n",
    "\n",
    "    extractor = Model(\n",
    "        inputs=base_model.input,\n",
    "        outputs=output_features,\n",
    "        name=f\"{backbone_name}_AttentionExtractor\",\n",
    "    )\n",
    "    return extractor\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### File 2: `utils.py`\n",
    "\n",
    "This file contains all helper functions: data path loading, the feature extraction/stacking function, and all the feature selection algorithms (ReliefF, mRMR, etc.)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "%%writefile utils.py\n",
    "import os\n",
    "import glob\n",
    "import numpy as np\n",
    "import tensorflow.keras.preprocessing.image as tf_image\n",
    "from sklearn.utils import shuffle\n",
    "from sklearn.feature_selection import mutual_info_classif\n",
    "from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.neighbors import KNeighborsClassifier\n",
    "\n",
    "# --- Data Loading ---\n",
    "\n",
    "def prepare_dataset_paths(base_path, train_counts, val_counts, test_counts):\n",
    "    \"\"\"Prepare dataset paths and labels based on subfolders.\"\"\"\n",
    "    print(f\"Loading data paths from: {base_path}\")\n",
    "    \n",
    "    def get_paths_labels(split, real_count, fake_count):\n",
    "        # Handle cases where counts are not provided (e.g., test split in train.py)\n",
    "        real_paths = sorted(glob.glob(os.path.join(base_path, split, \"real\", \"*.*\")))\n",
    "        fake_paths = sorted(glob.glob(os.path.join(base_path, split, \"fake\", \"*.*\")))\n",
    "        \n",
    "        if real_count > 0:\n",
    "            real_paths = real_paths[:real_count]\n",
    "        if fake_count > 0:\n",
    "            fake_paths = fake_paths[:fake_count]\n",
    "\n",
    "        paths = real_paths + fake_paths\n",
    "        labels = [0] * len(real_paths) + [1] * len(fake_paths)\n",
    "        \n",
    "        if not paths:\n",
    "            print(f\"  Loaded {split}: 0 paths.\")\n",
    "            return [], []\n",
    "\n",
    "        paths, labels = shuffle(paths, labels, random_state=42)\n",
    "        print(f\"  Loaded {split}: {len(real_paths)} real, {len(fake_paths)} fake. Total: {len(paths)}\")\n",
    "        return paths, labels\n",
    "\n",
    "    data = {}\n",
    "    if train_counts:\n",
    "        data[\"train\"] = get_paths_labels(\"train\", train_counts.get('real', 0), train_counts.get('fake', 0))\n",
    "    if val_counts:\n",
    "        data[\"val\"] = get_paths_labels(\"val\", val_counts.get('real', 0), val_counts.get('fake', 0))\n",
    "    if test_counts:\n",
    "        data[\"test\"] = get_paths_labels(\"test\", test_counts.get('real', 0), test_counts.get('fake', 0))\n",
    "    \n",
    "    return data\n",
    "\n",
    "\n",
    "# --- Feature Extraction ---\n",
    "\n",
    "def extract_features(file_paths, extractors, preprocessors, input_shape=(299, 299, 3), batch_size=32):\n",
    "    \"\"\"\n",
    "    Extracts features from all three attention-based extractors and stacks them.\n",
    "    \"\"\"\n",
    "    all_features = {name: [] for name in extractors.keys()}\n",
    "\n",
    "    for i in range(0, len(file_paths), batch_size):\n",
    "        if i % (batch_size * 10) == 0:\n",
    "            print(f\"  Processing batch {i // batch_size} / {len(file_paths) // batch_size}\")\n",
    "            \n",
    "        batch_paths = file_paths[i : i + batch_size]\n",
    "        loaded_images = []\n",
    "        valid_paths_idx = []\n",
    "\n",
    "        for idx, img_path in enumerate(batch_paths):\n",
    "            try:\n",
    "                image = tf_image.load_img(\n",
    "                    img_path, target_size=input_shape[:2]\n",
    "                )\n",
    "                image = tf_image.img_to_array(image)\n",
    "                loaded_images.append(image)\n",
    "                valid_paths_idx.append(idx)\n",
    "            except Exception as e:\n",
    "                print(f\"Warning: Error loading {img_path}: {e}\")\n",
    "                \n",
    "        if not loaded_images:\n",
    "            continue\n",
    "\n",
    "        loaded_images = np.array(loaded_images)\n",
    "\n",
    "        for name, extractor in extractors.items():\n",
    "            preprocess_func = preprocessors[name]\n",
    "            preprocessed_batch = preprocess_func(loaded_images.copy())\n",
    "            features = extractor.predict(preprocessed_batch, verbose=0)\n",
    "            all_features[name].extend(features)\n",
    "\n",
    "    print(\"Stacking extracted features...\")\n",
    "    stacked_features = np.concatenate(\n",
    "        [\n",
    "            np.array(all_features[\"DenseNet121\"]),\n",
    "            np.array(all_features[\"EfficientNetB0\"]),\n",
    "            np.array(all_features[\"Xception\"]),\n",
    "        ],\n",
    "        axis=1,\n",
    "    )\n",
    "    return stacked_features\n",
    "\n",
    "\n",
    "# --- Feature Selection (from feature-selection-aided...ipynb) ---\n",
    "\n",
    "def relief_f_score(X, y, k=10):\n",
    "    \"\"\"ReliefF feature selection algorithm\"\"\"\n",
    "    print(\"  Running ReliefF...\")\n",
    "    n_samples, n_features = X.shape\n",
    "    feature_scores = np.zeros(n_features)\n",
    "\n",
    "    for i in range(n_samples):\n",
    "        if i % 100 == 0:\n",
    "            print(f\"    ReliefF processing sample {i} / {n_samples}\")\n",
    "        distances = np.sum((X - X[i]) ** 2, axis=1)\n",
    "        nearest_indices = np.argsort(distances)[1 : k + 1]\n",
    "        hits = [idx for idx in nearest_indices if y[idx] == y[i]]\n",
    "        misses = [idx for idx in nearest_indices if y[idx] != y[i]]\n",
    "\n",
    "        for j in range(n_features):\n",
    "            if hits:\n",
    "                hit_diff = np.mean([abs(X[i, j] - X[idx, j]) for idx in hits])\n",
    "                feature_scores[j] -= hit_diff\n",
    "            if misses:\n",
    "                miss_diff = np.mean([abs(X[i, j] - X[idx, j]) for idx in misses])\n",
    "                feature_scores[j] += miss_diff\n",
    "    \n",
    "    min_score = np.min(feature_scores)\n",
    "    max_score = np.max(feature_scores)\n",
    "    if max_score == min_score:\n",
    "        return np.zeros_like(feature_scores)\n",
    "    return (feature_scores - min_score) / (max_score - min_score)\n",
    "\n",
    "def mrmr_score(X, y, selected_features=None):\n",
    "    \"\"\"Minimum Redundancy Maximum Relevance score helper\"\"\"\n",
    "    if selected_features is None:\n",
    "        selected_features = []\n",
    "\n",
    "    mi_scores = mutual_info_classif(X, y, random_state=42)\n",
    "\n",
    "    if not selected_features:\n",
    "        return mi_scores\n",
    "\n",
    "    n_features = X.shape[1]\n",
    "    mrmr_scores = np.zeros(n_features)\n",
    "\n",
    "    for i in range(n_features):\n",
    "        if i in selected_features:\n",
    "            mrmr_scores[i] = -np.inf\n",
    "            continue\n",
    "        \n",
    "        relevance = mi_scores[i]\n",
    "        \n",
    "        if selected_features:\n",
    "            redundancy = np.mean(\n",
    "                [\n",
    "                    mutual_info_classif(X[:, [i, j]], y, random_state=42)[0]\n",
    "                    for j in selected_features\n",
    "                ]\n",
    "            )\n",
    "        else:\n",
    "            redundancy = 0\n",
    "        \n",
    "        mrmr_scores[i] = relevance - redundancy\n",
    "    return mrmr_scores\n",
    "\n",
    "def calculate_fitness(X_train, y_train, X_val, y_val, feature_indices, weight=0.9):\n",
    "    \"\"\"Calculate fitness function for feature selection\"\"\"\n",
    "    if len(feature_indices) == 0:\n",
    "        return 0\n",
    "    \n",
    "    X_train_sel = X_train[:, feature_indices]\n",
    "    X_val_sel = X_val[:, feature_indices]\n",
    "    \n",
    "    temp_scaler = StandardScaler()\n",
    "    X_train_scaled = temp_scaler.fit_transform(X_train_sel)\n",
    "    X_val_scaled = temp_scaler.transform(X_val_sel)\n",
    "    \n",
    "    temp_knn = KNeighborsClassifier(n_neighbors=5)\n",
    "    temp_knn.fit(X_train_scaled, y_train)\n",
    "    accuracy = temp_knn.score(X_val_scaled, y_val)\n",
    "    \n",
    "    feature_ratio = len(feature_indices) / X_train.shape[1]\n",
    "    fitness = weight * accuracy + (1 - weight) * (1 - feature_ratio)\n",
    "    return fitness\n",
    "\n",
    "def feature_selection(\n",
    "    X_train, y_train, X_val, y_val, tau=0.3, alpha=0.1, beta=0.1, max_iterations=250\n",
    "):\n",
    "    \"\"\"Combined feature selection with ReliefF, MI, mRMR and inclusion-exclusion\"\"\"\n",
    "    print(\"Starting feature selection...\")\n",
    "    \n",
    "    relief_scores = relief_f_score(X_train, y_train)\n",
    "    \n",
    "    print(\"  Running Mutual Information...\")\n",
    "    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)\n",
    "    \n",
    "    print(\"  Running mRMR...\")\n",
    "    mrmr_scores = mrmr_score(X_train, y_train)\n",
    "\n",
    "    mi_scores = (mi_scores - np.min(mi_scores)) / (np.max(mi_scores) - np.min(mi_scores))\n",
    "    mrmr_scores = (mrmr_scores - np.min(mrmr_scores)) / (np.max(mrmr_scores) - np.min(mrmr_scores))\n",
    "\n",
    "    combined_scores = (relief_scores + mi_scores + mrmr_scores) / 3\n",
    "    \n",
    "    n_features = len(combined_scores)\n",
    "    n_initial = int(tau * n_features)\n",
    "    initial_indices = np.argsort(combined_scores)[-n_initial:]\n",
    "\n",
    "    print(f\"  Initial selection: {len(initial_indices)} features\")\n",
    "    \n",
    "    best_features = initial_indices.copy()\n",
    "    best_fitness = calculate_fitness(X_train, y_train, X_val, y_val, best_features)\n",
    "    print(f\"  Initial fitness: {best_fitness:.4f}\")\n",
    "\n",
    "    for iteration in range(max_iterations):\n",
    "        current_features = best_features.copy()\n",
    "        \n",
    "        n_exclude = max(1, int(alpha * len(current_features)))\n",
    "        if len(current_features) > n_exclude:\n",
    "            exclude_indices = np.random.choice(\n",
    "                len(current_features), n_exclude, replace=False\n",
    "            )\n",
    "            current_features = np.delete(current_features, exclude_indices)\n",
    "        \n",
    "        remaining_features = np.setdiff1d(np.arange(n_features), current_features)\n",
    "        if len(remaining_features) > 0:\n",
    "            n_include = min(int(beta * n_features), len(remaining_features))\n",
    "            remaining_scores = combined_scores[remaining_features]\n",
    "            include_indices = remaining_features[\n",
    "                np.argsort(remaining_scores)[-n_include:]\n",
    "            ]\n",
    "            current_features = np.unique(np.concatenate([current_features, include_indices]))\n",
    "        \n",
    "        current_fitness = calculate_fitness(X_train, y_train, X_val, y_val, current_features)\n",
    "\n",
    "        if current_fitness > best_fitness:\n",
    "            best_features = current_features.copy()\n",
    "            best_fitness = current_fitness\n",
    "            if iteration % 10 == 0:\n",
    "                print(\n",
    "                    f\"    Iter {iteration}: New best fitness = {best_fitness:.4f}, Features = {len(best_features)}\"\n",
    "                )\n",
    "    \n",
    "    print(f\"Feature selection completed. Selected {len(best_features)} features.\")\n",
    "    return best_features\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### File 3: `train.py`\n",
    "\n",
    "This file defines the `train` class. Its `run` method executes the full pipeline:\n",
    "1.  Load data paths (`utils.py`).\n",
    "2.  Create the 3 feature extractor models (`model.py`).\n",
    "3.  Extract features from train/val sets (`utils.py`).\n",
    "4.  Run feature selection (`utils.py`).\n",
    "5.  Train the final KNN and Scaler.\n",
    "6.  Save the `knn_model.joblib`, `scaler.joblib`, and `selected_features.joblib` for `predict.py` to use."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "%%writefile train.py\n",
    "import os\n",
    "import numpy as np\n",
    "import joblib\n",
    "from sklearn.neighbors import KNeighborsClassifier\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "import tensorflow as tf\n",
    "\n",
    "import model\n",
    "import utils\n",
    "\n",
    "# Set random seeds for reproducibility\n",
    "np.random.seed(42)\n",
    "tf.random.set_seed(42)\n",
    "\n",
    "class train:\n",
    "    def __init__(self, train_path, val_path, train_counts, val_counts):\n",
    "        self.train_path = train_path\n",
    "        self.val_path = val_path\n",
    "        self.train_counts = train_counts\n",
    "        self.val_counts = val_counts\n",
    "        self.model_save_dir = \"models\"\n",
    "        os.makedirs(self.model_save_dir, exist_ok=True)\n",
    "\n",
    "    def run(self, tau, alpha, beta, max_iter, batch_size=32):\n",
    "        print(\"--- Starting Training Pipeline ---\")\n",
    "        \n",
    "        # 1. Load Data Paths\n",
    "        train_data = utils.prepare_dataset_paths(\n",
    "            self.train_path, self.train_counts, {}, {}\n",
    "        )\n",
    "        val_data = utils.prepare_dataset_paths(\n",
    "            self.val_path, {}, self.val_counts, {}\n",
    "        )\n",
    "        \n",
    "        train_paths, train_labels = train_data[\"train\"]\n",
    "        val_paths, val_labels = val_data[\"val\"]\n",
    "\n",
    "        # 2. Create Feature Extractors\n",
    "        print(\"Loading feature extractor models...\")\n",
    "        model_names = [\"DenseNet121\", \"EfficientNetB0\", \"Xception\"]\n",
    "        extractors = {}\n",
    "        preprocessors = {}\n",
    "        input_shape = (299, 299, 3)\n",
    "        \n",
    "        for name in model_names:\n",
    "            print(f\"  Creating {name} extractor...\")\n",
    "            extractors[name] = model.create_attention_extractor(name, input_shape)\n",
    "            _, preproc, _, _, _ = model.get_backbone_config(name, input_shape)\n",
    "            preprocessors[name] = preproc\n",
    "        \n",
    "        # 3. Extract Features\n",
    "        print(\"\\nExtracting Training Features... (This may take a while)\")\n",
    "        train_features = utils.extract_features(\n",
    "            train_paths, extractors, preprocessors, input_shape, batch_size\n",
    "        )\n",
    "        print(\"\\nExtracting Validation Features...\")\n",
    "        val_features = utils.extract_features(\n",
    "            val_paths, extractors, preprocessors, input_shape, batch_size\n",
    "        )\n",
    "        print(f\"\\nTotal stacked feature dimension: {train_features.shape[1]}\")\n",
    "\n",
    "        # 4. Run Feature Selection\n",
    "        selected_indices = utils.feature_selection(\n",
    "            train_features, train_labels, \n",
    "            val_features, val_labels, \n",
    "            tau, alpha, beta, max_iter\n",
    "        )\n",
    "\n",
    "        # 5. Train Final Classifier\n",
    "        print(\"Training final KNN classifier...\")\n",
    "        knn = KNeighborsClassifier(n_neighbors=5)\n",
    "        scaler = StandardScaler()\n",
    "        \n",
    "        X_train_selected = train_features[:, selected_indices]\n",
    "        X_train_scaled = scaler.fit_transform(X_train_selected)\n",
    "        knn.fit(X_train_scaled, train_labels)\n",
    "\n",
    "        # 6. Save models to disk\n",
    "        print(\"Saving models to disk...\")\n",
    "        joblib.dump(knn, os.path.join(self.model_save_dir, 'knn_model.joblib'))\n",
    "        joblib.dump(scaler, os.path.join(self.model_save_dir, 'scaler.joblib'))\n",
    "        joblib.dump(selected_indices, os.path.join(self.model_save_dir, 'selected_features.joblib'))\n",
    "        \n",
    "        print(\"--- Training Pipeline Complete ---\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### File 4: `main.py`\n",
    "\n",
    "This is the main entry point that parses command-line arguments and calls the `train` class."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "%%writefile main.py\n",
    "import argparse\n",
    "from train import train\n",
    "\n",
    "def main():\n",
    "    parser = argparse.ArgumentParser(description='Train Combined Deepfake Detector')\n",
    "\n",
    "    # Paths\n",
    "    parser.add_argument('--train_path', type=str, required=True, help='Base path to training dataset')\n",
    "    parser.add_argument('--val_path', type=str, required=True, help='Base path to validation dataset')\n",
    "\n",
    "    # Data counts\n",
    "    parser.add_argument('--train_real', type=int, default=2930, help='Num real training images')\n",
    "    parser.add_argument('--train_fake', type=int, default=2946, help='Num fake training images')\n",
    "    parser.add_argument('--val_real', type=int, default=200, help='Num real validation images')\n",
    "    parser.add_argument('--val_fake', type=int, default=200, help='Num fake validation images')\n",
    "\n",
    "    # Feature selection params\n",
    "    parser.add_argument('--tau', type=float, default=0.3, help='Initial selection threshold')\n",
    "    parser.add_argument('--alpha', type=float, default=0.1, help='Exclusion percentage')\n",
    "    parser.add_argument('--beta', type=float, default=0.1, help='Inclusion percentage')\n",
    "    parser.add_argument('--max_iter', type=int, default=250, help='Optimization iterations')\n",
    "    parser.add_argument('--batch_size', type=int, default=32, help='Batch size for feature extraction')\n",
    "\n",
    "    args = parser.parse_args()\n",
    "\n",
    "    train_counts = {'real': args.train_real, 'fake': args.train_fake}\n",
    "    val_counts = {'real': args.val_real, 'fake': args.val_fake}\n",
    "\n",
    "    print(\"--- Configuration ---\")\n",
    "    print(f\"Train Path: {args.train_path}\")\n",
    "    print(f\"Val Path: {args.val_path}\")\n",
    "    print(f\"Train Counts: {train_counts}\")\n",
    "    print(f\"Val Counts: {val_counts}\")\n",
    "    print(f\"FS Params: tau={args.tau}, alpha={args.alpha}, beta={args.beta}, iter={args.max_iter}\")\n",
    "    print(\"---------------------\\n\")\n",
    "\n",
    "    trainer = train(args.train_path, args.val_path, train_counts, val_counts)\n",
    "    trainer.run(args.tau, args.alpha, args.beta, args.max_iter, args.batch_size)\n",
    "\n",
    "if __name__ == '__main__':\n",
    "    main()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### File 5: `predict.py`\n",
    "\n",
    "This script is for evaluation. It loads the models saved by `train.py`, extracts features from the test set, and prints the final performance."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "%%writefile predict.py\n",
    "import argparse\n",
    "import joblib\n",
    "import os\n",
    "import numpy as np\n",
    "import tensorflow as tf\n",
    "from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix\n",
    "\n",
    "import model\n",
    "import utils\n",
    "\n",
    "# Set random seeds for reproducibility\n",
    "np.random.seed(42)\n",
    "tf.random.set_seed(42)\n",
    "\n",
    "def main():\n",
    "    parser = argparse.ArgumentParser(description='Evaluate Combined Deepfake Detector')\n",
    "    \n",
    "    # Paths\n",
    "    parser.add_argument('--test_path', type=str, required=True, help='Base path to test dataset')\n",
    "    parser.add_argument('--model_dir', type=str, default='models', help='Directory containing saved models')\n",
    "\n",
    "    # Data counts\n",
    "    parser.add_argument('--test_real', type=int, default=100, help='Num real test images')\n",
    "    parser.add_argument('--test_fake', type=int, default=100, help='Num fake test images')\n",
    "    parser.add_argument('--batch_size', type=int, default=32, help='Batch size for feature extraction')\n",
    "\n",
    "    args = parser.parse_args()\n",
    "\n",
    "    print(\"--- Starting Prediction Pipeline ---\")\n",
    "\n",
    "    # 1. Load saved models\n",
    "    print(\"Loading KNN, Scaler, and Feature List...\")\n",
    "    try:\n",
    "        knn = joblib.load(os.path.join(args.model_dir, 'knn_model.joblib'))\n",
    "        scaler = joblib.load(os.path.join(args.model_dir, 'scaler.joblib'))\n",
    "        selected_indices = joblib.load(os.path.join(args.model_dir, 'selected_features.joblib'))\n",
    "    except FileNotFoundError as e:\n",
    "        print(f\"Error: Model file not found. {e}\")\n",
    "        print(\"Please run train.py first to generate the models.\")\n",
    "        return\n",
    "\n",
    "    # 2. Create Feature Extractors\n",
    "    print(\"Loading feature extractor models...\")\n",
    "    model_names = [\"DenseNet121\", \"EfficientNetB0\", \"Xception\"]\n",
    "    extractors = {}\n",
    "    preprocessors = {}\n",
    "    input_shape = (299, 299, 3)\n",
    "    \n",
    "    for name in model_names:\n",
    "        extractors[name] = model.create_attention_extractor(name, input_shape)\n",
    "        _, preproc, _, _, _ = model.get_backbone_config(name, input_shape)\n",
    "        preprocessors[name] = preproc\n",
    "\n",
    "    # 3. Load Test Data Paths\n",
    "    test_counts = {'real': args.test_real, 'fake': args.test_fake}\n",
    "    test_data = utils.prepare_dataset_paths(\n",
    "        args.test_path, {}, {}, test_counts\n",
    "    )\n",
    "    test_paths, test_labels = test_data[\"test\"]\n",
    "    if not test_paths:\n",
    "        print(f\"No test images found in {args.test_path}. Exiting.\")\n",
    "        return\n",
    "\n",
    "    # 4. Extract Test Features\n",
    "    print(\"\\nExtracting Test Features... (This may take a while)\")\n",
    "    test_features = utils.extract_features(\n",
    "        test_paths, extractors, preprocessors, input_shape, args.batch_size\n",
    "    )\n",
    "\n",
    "    # 5. Apply Selection and Scaling\n",
    "    print(\"Applying feature selection and scaling...\")\n",
    "    X_test_selected = test_features[:, selected_indices]\n",
    "    X_test_scaled = scaler.transform(X_test_selected)\n",
    "\n",
    "    # 6. Run Prediction\n",
    "    print(\"Running predictions...\")\n",
    "    predictions = knn.predict(X_test_scaled)\n",
    "    probabilities = knn.predict_proba(X_test_scaled)\n",
    "\n",
    "    # 7. Report Results\n",
    "    test_accuracy = accuracy_score(test_labels, predictions)\n",
    "    test_auc = roc_auc_score(test_labels, probabilities[:, 1])\n",
    "    cm = confusion_matrix(test_labels, predictions)\n",
    "\n",
    "    print(\"\\n--- FINAL TEST RESULTS ---\")\n",
    "    print(f\"  Test Accuracy: {test_accuracy:.4f}\")\n",
    "    print(f\"  Test AUC: {test_auc:.4f}\")\n",
    "    print(f\"  Original features: {test_features.shape[1]}\")\n",
    "    print(f\"  Selected features: {len(selected_indices)}\")\n",
    "    print(f\"  Feature reduction: {(1 - len(selected_indices)/test_features.shape[1])*100:.2f}%\")\n",
    "    print(\"  Confusion Matrix:\")\n",
    "    print(cm)\n",
    "    print(\"-------------------------\")\n",
    "\n",
    "if __name__ == '__main__':\n",
    "    main()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 2: Train the Model\n",
    "\n",
    "This cell executes the `main.py` script to start the training process. \n",
    "\n",
    "**This is a very long-running cell.** It will:\n",
    "1.  Extract features from all 3 models for all **train** images.\n",
    "2.  Extract features from all 3 models for all **validation** images.\n",
    "3.  Run the computationally expensive feature selection algorithms (ReliefF, MI, mRMR).\n",
    "4.  Run the 250-iteration inclusion-exclusion optimization.\n",
    "5.  Train and save the final KNN model, scaler, and feature list.\n",
    "\n",
    "I'm using the paths and counts from your `feature-selection-aided...` notebook."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python main.py \\\n",
    "    --train_path \"/kaggle/input/faceforencispp-extracted-frames/\" \\\n",
    "    --val_path \"/kaggle/input/faceforencispp-extracted-frames/\" \\\n",
    "    --train_real 2930 \\\n",
    "    --train_fake 2946 \\\n",
    "    --val_real 200 \\\n",
    "    --val_fake 200 \\\n",
    "    --max_iter 250"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 3: Run Prediction\n",
    "\n",
    "After training is complete, this cell runs the `predict.py` script. It will load the saved models (`knn_model.joblib`, etc.) and evaluate them on the test set."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python predict.py \\\n",
    "    --test_path \"/kaggle/input/faceforencispp-extracted-frames/\" \\\n",
    "    --test_real 100 \\\n",
    "    --test_fake 100"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.13"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
